In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import xarray as xr

In [18]:
from marine_qc import (
    do_hard_limit_check,
    do_iquam_track_check,
    do_position_check,
    do_supersaturation_check,
    do_time_check,
)

In [11]:
lat = pd.Series([10.0, 200.0, 15.0])
lon = pd.Series([400.0, 10.0, 15.0])
# hour = pd.Series([12.25, 12.50, -12.75])
date = pd.Series(["2026-07-27T12:15", "2026-07-12T12:30", "2026-07-12T25:00"])
# data = pd.DataFrame({
#    "lat": lat,
#    "lon": lon,
#    "hour": hour,
#    "date": date,
# })
temperature = np.random.uniform(250, 300, size=(len(date), len(lat), len(lon)))
da_temp = xr.DataArray(
    temperature,
    dims=["date", "lat", "lon"],
    coords={"date": date, "lat": lat, "lon": lon},
)
dewpoint = np.random.uniform(250, 300, size=(len(date), len(lat), len(lon)))
da_dew = xr.DataArray(
    dewpoint,
    dims=["date", "lat", "lon"],
    coords={"date": date, "lat": lat, "lon": lon},
)
ds = xr.Dataset(
    data_vars={
        "at2": da_temp,
        "dpt": da_dew,
    }
)
ds

<xarray.Dataset> Size: 504B
Dimensions:  (date: 3, lat: 3, lon: 3)
Coordinates:
  * date     (date) object 24B '2026-07-27T12:15' ... '2026-07-12T25:00'
  * lat      (lat) float64 24B 10.0 200.0 15.0
  * lon      (lon) float64 24B 400.0 10.0 15.0
Data variables:
    at2      (date, lat, lon) float64 216B 295.2 291.2 272.8 ... 266.7 299.6
    dpt      (date, lat, lon) float64 216B 298.6 257.3 254.0 ... 263.9 257.8

In [5]:
da_temp

<xarray.DataArray (date: 3, lat: 3, lon: 3)> Size: 216B
array([[[264.98691644, 262.85428385, 251.08655994],
        [275.88759183, 296.53272415, 280.09177198],
        [269.29604752, 262.90142777, 298.38260674]],

       [[281.42111417, 274.69200398, 294.72952423],
        [259.25234855, 263.40656731, 251.91206633],
        [272.9325358 , 264.26434135, 265.3874637 ]],

       [[285.31029301, 250.90396187, 272.24027344],
        [298.07667713, 278.90229419, 283.23144503],
        [268.54322383, 250.78802845, 288.92408568]]])
Coordinates:
  * date     (date) object 24B '2026-07-27T12:15' ... '2026-07-12T25:00'
  * lat      (lat) float64 24B 10.0 200.0 15.0
  * lon      (lon) float64 24B 400.0 10.0 15.0

In [6]:
qc_hard = do_hard_limit_check(da_temp, limits=(255, 295))
qc_hard

<xarray.DataArray (date: 3, lat: 3, lon: 3)> Size: 216B
array([[[0, 0, 1],
        [0, 1, 0],
        [0, 0, 1]],

       [[0, 0, 0],
        [0, 0, 1],
        [0, 0, 0]],

       [[0, 1, 0],
        [1, 0, 0],
        [0, 1, 0]]])
Coordinates:
  * date     (date) object 24B '2026-07-27T12:15' ... '2026-07-12T25:00'
  * lat      (lat) float64 24B 10.0 200.0 15.0
  * lon      (lon) float64 24B 400.0 10.0 15.0

In [25]:
do_hard_limit_check(value=da_temp, limits=(255, 295))

<xarray.DataArray (date: 3, lat: 3, lon: 3)> Size: 216B
array([[[1, 0, 0],
        [0, 0, 1],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 1, 1],
        [0, 0, 1]]])
Coordinates:
  * date     (date) object 24B '2026-07-27T12:15' ... '2026-07-12T25:00'
  * lat      (lat) float64 24B 10.0 200.0 15.0
  * lon      (lon) float64 24B 400.0 10.0 15.0

In [7]:
qc_pos = do_position_check(data=da_temp)
qc_pos

<xarray.DataArray (lat: 3)> Size: 24B
array([1, 1, 0])
Coordinates:
  * lat      (lat) float64 24B 10.0 200.0 15.0

In [8]:
qc_time = do_time_check(data=da_temp)
qc_time

C:\Users\llierham\mobaxterm\github_ll\marine_qc\src\marine_qc\helpers\time_control.py:80: UserWarning: Could not convert '2026-07-12T25:00' to datetime: hour must be in 0..23: 2026-07-12T25:00
  extracted_list: list[dict[str, float]] = [split_date(d) for d in date]


<xarray.DataArray (date: 3)> Size: 24B
array([0, 0, 2])
Coordinates:
  * date     (date) object 24B '2026-07-27T12:15' ... '2026-07-12T25:00'

In [9]:
do_supersaturation_check(data=ds)

<xarray.DataArray (date: 3, lat: 3, lon: 3)> Size: 216B
array([[[1, 1, 1],
        [1, 0, 0],
        [1, 1, 0]],

       [[1, 1, 0],
        [1, 1, 1],
        [0, 0, 0]],

       [[0, 1, 0],
        [0, 1, 0],
        [0, 1, 0]]])
Coordinates:
  * date     (date) object 24B '2026-07-27T12:15' ... '2026-07-12T25:00'
  * lat      (lat) float64 24B 10.0 200.0 15.0
  * lon      (lon) float64 24B 400.0 10.0 15.0

In [22]:
date = pd.date_range(start="2023-01-01T12:00:00", end="2023-01-07T12:00:00", freq="D")
latitude = [0.0, 5.0, 10.0, 15.0, 20.0, 25.0, 30.0]  # Example latitudes
longitude = [0.0, 10.0, 20.0, 30.0, 40.0, 50.0, 60.0]  # Example longitudes
temperature = np.random.rand(7) + 283.15  # Example temperatures

# Create the xarray Dataset
ds = xr.Dataset(
    {
        "lat": ("date", latitude),
        "lon": ("date", longitude),
        "temperature": ("date", temperature),
    },
    coords={"date": date},
)
ds

<xarray.Dataset> Size: 224B
Dimensions:      (date: 7)
Coordinates:
  * date         (date) datetime64[us] 56B 2023-01-01T12:00:00 ... 2023-01-07...
Data variables:
    lat          (date) float64 56B 0.0 5.0 10.0 15.0 20.0 25.0 30.0
    lon          (date) float64 56B 0.0 10.0 20.0 30.0 40.0 50.0 60.0
    temperature  (date) float64 56B 283.3 283.4 283.4 283.2 284.1 283.6 283.6

In [23]:
do_iquam_track_check(
    data=ds,
    speed_limit=10.0,
    delta_d=1.11,
    delta_t=0.01,
    n_neighbours=5,
)

<xarray.DataArray (date: 7)> Size: 56B
array([0, 1, 1, 1, 1, 1, 0])
Coordinates:
  * date     (date) datetime64[us] 56B 2023-01-01T12:00:00 ... 2023-01-07T12:...

In [24]:
ds.attrs

{}